# Pinecone

**Pinecone**은 임베딩(embedding) 벡터를 저장하고, 의미적으로 유사한 데이터를 빠르게 검색할 수 있도록 설계된  
**클라우드 기반의 고성능 벡터 데이터베이스(vector database)**.


주로 **RAG(Retrieval-Augmented Generation)**, semantic search, 추천 시스템 등 AI 서비스에서 활용됨.

---

## 왜 필요한가?

LLM은 외부 문서를 직접 기억하지 못하기 때문에, 다음과 같은 구조를 사용함.

1. 문서를 작은 단위(chunk)로 나눈다  
2. 각 문장을 벡터(embedding)로 변환한다  
3. 벡터를 데이터베이스에 저장한다  
4. 질문도 벡터로 변환한다  
5. 가장 유사한 벡터를 검색해 관련 문서를 가져옴

이 과정에서 Pinecone은 **“질문과 가장 관련 있는 문서를 빠르게 찾아주는 역할”**을 수행함.

---

## 로컬 벡터 DB와의 차이

대표적인 로컬 벡터 DB:
- FAISS
- Chroma

이들은 다음과 같은 특징을 가짐.

### 장점
- 무료, 오픈소스
- 실험 및 연구에 적합
- 설치와 사용이 간단

### 한계
- 대용량 데이터에서 검색 속도 저하
- 서버 확장 및 운영 관리가 어려움
- 멀티 사용자 환경 대응이 제한적

---

## Pinecone의 강점

Pinecone은 이러한 한계를 해결하기 위해 설계된 **클라우드 기반 서비스**.

### 핵심 특징

- **고속 유사도 검색**
  - 대규모 벡터에서도 빠른 검색 성능 유지

- **확장성 (Scalability)**
  - 데이터가 많아져도 자동으로 확장 가능

- **Namespace 기반 데이터 분리**
  - 사용자별 / 프로젝트별 데이터 관리 가능

- **Metadata Filtering**
  - 예: `topic="ct"`, `source="paper"` 조건 검색

- **운영 부담 없음**
  - 서버 관리 없이 API로 바로 사용 가능

---

## RAG에서 Pinecone의 역할

RAG 구조에서 Pinecone은 다음을 담당함.

- 문서 embedding 저장
- 질문과 유사한 문서 검색 (retriever)
- 특정 조건 기반 필터링 검색

LLM이 답변을 생성하기 전에 **“어떤 문서를 참고해야 하는지 결정해주는 핵심 컴포넌트”**.

In [1]:
# 패키지 설치
!pip -q install openai pinecone tiktoken pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.8/742.8 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.7/280.7 kB 31.8 MB/s eta 0:00:00


In [2]:
# API key 입력
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")
os.environ["PINECONE_API_KEY"] = getpass("Pinecone API Key: ")

OpenAI API Key: ··········
Pinecone API Key: ··········


In [3]:
# 기본 라이브러리 import
import os
import time
import uuid
import math
import pandas as pd

from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec

In [4]:
# OpenAI, Pinecone 클라이언트 생성
openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

In [5]:
# 실습용 문서 준비
documents = [
    {
        "id": "doc1",
        "text": """
        Computed tomography (CT) reconstruction aims to recover cross-sectional images from projection data.
        Filtered back projection (FBP) is fast and widely used, but it can produce streak artifacts when the data are noisy or incomplete.
        Iterative methods often improve image quality at the cost of increased computation.
        """,
        "metadata": {"topic": "ct", "source": "lecture", "level": "basic"}
    },
    {
        "id": "doc2",
        "text": """
        In sparse-view CT, only a limited number of projection angles are available.
        This reduces radiation dose, but makes reconstruction more ill-posed.
        Regularization and learned priors are commonly used to suppress artifacts.
        """,
        "metadata": {"topic": "ct", "source": "paper", "level": "intermediate"}
    },
    {
        "id": "doc3",
        "text": """
        A vector database stores embeddings and retrieves semantically similar items.
        In retrieval-augmented generation, relevant chunks are retrieved first and then given to a language model as context.
        Chunking quality strongly affects retrieval performance.
        """,
        "metadata": {"topic": "rag", "source": "lecture", "level": "basic"}
    },
    {
        "id": "doc4",
        "text": """
        Pinecone supports dense and sparse indexes.
        Dense indexes are used for semantic search, while sparse indexes are useful for lexical search.
        Namespaces can isolate data for multitenancy or logical partitioning.
        """,
        "metadata": {"topic": "pinecone", "source": "docs", "level": "intermediate"}
    },
    {
        "id": "doc5",
        "text": """
        Metadata filtering is useful when you want to restrict retrieval to a subset of documents,
        such as papers only, lecture notes only, or a specific domain.
        This often improves precision in practical RAG systems.
        """,
        "metadata": {"topic": "rag", "source": "paper", "level": "intermediate"}
    },
]

In [6]:
# 간단한 chunking 함수
def chunk_text(text, chunk_size=220, overlap=40):
    text = " ".join(text.split())
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        if end >= len(text):
            break
        start = end - overlap

    return chunks

In [7]:
# 문서를 chunk 단위로 펼치기
chunked_docs = []

for doc in documents:
    chunks = chunk_text(doc["text"], chunk_size=180, overlap=30)
    for i, chunk in enumerate(chunks):
        chunked_docs.append({
            "id": f'{doc["id"]}_chunk{i}',
            "text": chunk,
            "metadata": {
                **doc["metadata"],
                "doc_id": doc["id"],
                "chunk_id": i
            }
        })

len(chunked_docs), chunked_docs[:2]

(10,
 [{'id': 'doc1_chunk0',
   'text': 'Computed tomography (CT) reconstruction aims to recover cross-sectional images from projection data. Filtered back projection (FBP) is fast and widely used, but it can produce stre',
   'metadata': {'topic': 'ct',
    'source': 'lecture',
    'level': 'basic',
    'doc_id': 'doc1',
    'chunk_id': 0}},
  {'id': 'doc1_chunk1',
   'text': ' used, but it can produce streak artifacts when the data are noisy or incomplete. Iterative methods often improve image quality at the cost of increased computation.',
   'metadata': {'topic': 'ct',
    'source': 'lecture',
    'level': 'basic',
    'doc_id': 'doc1',
    'chunk_id': 1}}])

In [8]:
# 임베딩 함수
## OpenAI 임베딩 API는 문자열 또는 문자열 배열 입력을 받아 벡터를 반환

EMBED_MODEL = "text-embedding-3-small"

def embed_texts(texts):
    response = openai_client.embeddings.create(
        model=EMBED_MODEL,
        input=texts
    )
    return [item.embedding for item in response.data]

In [9]:
# 임베딩 차원 확인
## Pinecone 인덱스를 만들 때는 임베딩 차원과 인덱스 차원이 반드시 맞아야 함

sample_vec = embed_texts(["hello pinecone"])[0]
embedding_dim = len(sample_vec)
print("embedding dimension:", embedding_dim)

embedding dimension: 1536


In [10]:
# Pinecone 인덱스 생성

INDEX_NAME = "colab-rag-pinecone-demo"

existing_indexes = [idx["name"] for idx in pc.list_indexes()]

if INDEX_NAME not in existing_indexes:
    pc.create_index(
        name=INDEX_NAME,
        dimension=embedding_dim,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print("Index created.")
else:
    print("Index already exists.")

Index created.


In [12]:
# 인덱스 준비 대기
while True:
    desc = pc.describe_index(INDEX_NAME)
    status = desc.status["ready"]
    print("ready:", status)
    if status:
        break
    time.sleep(5)

ready: True


In [13]:
# 인덱스 연결
index = pc.Index(INDEX_NAME)

- upsert: 벡터를 “삽입(insert) 또는 업데이트(update)”하는 작업
    - 같은 ID가 있으면 덮어쓰고, 없으면 새로 추가
- namespace: 데이터 묶음(폴더처럼 생각하면 됨)
    - 같은 인덱스 안에서 데이터를 구분하는 논리적 공간

In [14]:
# 벡터 업서트용 레코드 생성
## Pinecone은 namespace 단위 upsert를 지원하며, namespace는 데이터 분리와 멀티테년시에 유용
# namespace는 “벡터 데이터 폴더”이고, upsert할 때 그 폴더를 지정해서 데이터 섞임 없이 관리할 수 있다는 의미다.
# 멀티테년시(multi-tenancy): 하나의 시스템에서 여러 사용자/프로젝트를 운영할 때 사용
texts = [d["text"] for d in chunked_docs]
vectors = embed_texts(texts)

records = []
for d, vec in zip(chunked_docs, vectors):
    records.append({
        "id": d["id"],
        "values": vec,
        "metadata": {
            **d["metadata"],
            "text": d["text"]
        }
    })

records[:1]

[{'id': 'doc1_chunk0',
  'values': [0.0015935897827148438,
   -0.0009160041809082031,
   0.040802001953125,
   0.054046630859375,
   -0.03173828125,
   0.00222015380859375,
   -0.033355712890625,
   -0.0023212432861328125,
   -0.050079345703125,
   0.03399658203125,
   0.0172576904296875,
   0.017669677734375,
   0.0207061767578125,
   0.005084991455078125,
   0.025054931640625,
   0.00420379638671875,
   -0.0355224609375,
   -0.048095703125,
   -0.0201263427734375,
   -0.00591278076171875,
   0.0221405029296875,
   -0.02532958984375,
   0.0328369140625,
   -0.09112548828125,
   -0.03094482421875,
   0.021575927734375,
   0.0038051605224609375,
   -0.024871826171875,
   -0.01409912109375,
   -0.0010557174682617188,
   -0.06689453125,
   -0.034149169921875,
   -0.048431396484375,
   -0.0452880859375,
   -0.00426483154296875,
   0.0284271240234375,
   -0.025177001953125,
   0.063720703125,
   0.0192108154296875,
   0.041748046875,
   0.07769775390625,
   0.037872314453125,
   -0.01065826

In [15]:
# namespace별 upsert
## 일부는 reserach, 일부는 teaching namespace에 넣어보자.
research_records = []
teaching_records = []

for r in records:
    if r["metadata"]["source"] in ["paper", "docs"]:
        research_records.append(r)
    else:
        teaching_records.append(r)

index.upsert(vectors=research_records, namespace="research")
index.upsert(vectors=teaching_records, namespace="teaching")

print("research:", len(research_records))
print("teaching:", len(teaching_records))

research: 6
teaching: 4


In [16]:
# 인덱스 통계 확인
## Pinecone은 namespace별 벡터 수를 포함한 index stats를 제공
stats = index.describe_index_stats()
stats

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '209',
                                    'content-type': 'application/json',
                                    'date': 'Sun, 19 Apr 2026 12:05:18 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '35',
                                    'x-pinecone-request-latency-ms': '35',
                                    'x-pinecone-response-duration-ms': '36'}},
 'dimension': 1536,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'research': {'vector_count': 6},
                'teaching': {'vector_count': 4}},
 'storageFullness': 0.0,
 'total_vector_count': 10,
 'vector_type': 'dense'}

In [17]:
# 검색 함수 작성
##  Pinecone 검색은 top-k, metadata 포함 여부, namespace 지정 등을 조절할 수 있음
def search_pinecone(query, namespace=None, top_k=3, metadata_filter=None):
    qvec = embed_texts([query])[0]

    result = index.query(
        vector=qvec,
        top_k=top_k,
        namespace=namespace,
        include_metadata=True,
        filter=metadata_filter
    )
    return result

In [18]:
# 기본 검색 실습
result = search_pinecone(
    query="How can we reduce artifacts in sparse-view CT?",
    namespace="research",
    top_k=3
)

for i, match in enumerate(result["matches"], 1):
    print(f"[{i}] score={match['score']:.4f}")
    print("id:", match["id"])
    print("metadata:", {k: v for k, v in match["metadata"].items() if k != "text"})
    print("text:", match["metadata"]["text"])
    print("-" * 80)

[1] score=0.6861
id: doc2_chunk0
metadata: {'chunk_id': 0, 'doc_id': 'doc2', 'level': 'intermediate', 'source': 'paper', 'topic': 'ct'}
text: In sparse-view CT, only a limited number of projection angles are available. This reduces radiation dose, but makes reconstruction more ill-posed. Regularization and learned priors
--------------------------------------------------------------------------------
[2] score=0.4470
id: doc2_chunk1
metadata: {'chunk_id': 1, 'doc_id': 'doc2', 'level': 'intermediate', 'source': 'paper', 'topic': 'ct'}
text: ularization and learned priors are commonly used to suppress artifacts.
--------------------------------------------------------------------------------
[3] score=0.3187
id: doc5_chunk0
metadata: {'chunk_id': 0, 'doc_id': 'doc5', 'level': 'intermediate', 'source': 'paper', 'topic': 'rag'}
text: Metadata filtering is useful when you want to restrict retrieval to a subset of documents, such as papers only, lecture notes only, or a specific domain. This

In [19]:
# namespace 비교 실습
query = "What is retrieval-augmented generation?"

for ns in ["research", "teaching"]:
    print(f"\n===== namespace: {ns} =====")
    result = search_pinecone(query=query, namespace=ns, top_k=2)

    for i, match in enumerate(result["matches"], 1):
        print(f"[{i}] score={match['score']:.4f}")
        print(match["metadata"]["text"])
        print()


===== namespace: research =====
[1] score=0.2889
ularization and learned priors are commonly used to suppress artifacts.

[2] score=0.2840
in. This often improves precision in practical RAG systems.


===== namespace: teaching =====
[1] score=0.6030
A vector database stores embeddings and retrieves semantically similar items. In retrieval-augmented generation, relevant chunks are retrieved first and then given to a language mo

[2] score=0.3769
nd then given to a language model as context. Chunking quality strongly affects retrieval performance.



- 같은 질문이어도 namespace가 다르면 결과가 달라짐
- 서비스에서 사용자별/과목별/프로젝트별 분리가 쉬움

In [20]:
# metadata filter 실습
result = search_pinecone(
    query="vector database and chunking",
    namespace="research",
    top_k=5,
    metadata_filter={"topic": {"$eq": "rag"}}
)

for i, match in enumerate(result["matches"], 1):
    print(f"[{i}] score={match['score']:.4f}")
    print("topic:", match["metadata"]["topic"])
    print("source:", match["metadata"]["source"])
    print("text:", match["metadata"]["text"])
    print("-" * 80)

[1] score=0.2323
topic: rag
source: paper
text: in. This often improves precision in practical RAG systems.
--------------------------------------------------------------------------------
[2] score=0.2180
topic: rag
source: paper
text: Metadata filtering is useful when you want to restrict retrieval to a subset of documents, such as papers only, lecture notes only, or a specific domain. This often improves precis
--------------------------------------------------------------------------------


In [21]:
# 검색 결과 표로 보기

def result_to_df(result):
    rows = []
    for m in result["matches"]:
        rows.append({
            "id": m["id"],
            "score": m["score"],
            "topic": m["metadata"].get("topic"),
            "source": m["metadata"].get("source"),
            "level": m["metadata"].get("level"),
            "text": m["metadata"].get("text")
        })
    return pd.DataFrame(rows)

df = result_to_df(result)
df

,id,score,topic,source,level,text
0,doc5_chunk1,0.232316,rag,paper,intermediate,in. This often improves precision in practical...
1,doc5_chunk0,0.218049,rag,paper,intermediate,Metadata filtering is useful when you want to ...


In [25]:
# 간단한 RAG 답변 생성

CHAT_MODEL = "gpt-4.1-mini"

def generate_rag_answer(query, namespace=None, top_k=3, metadata_filter=None):
    result = search_pinecone(
        query=query,
        namespace=namespace,
        top_k=top_k,
        metadata_filter=metadata_filter
    )

    contexts = [m["metadata"]["text"] for m in result["matches"]]
    context_text = "\n\n".join([f"[Context {i+1}] {c}" for i, c in enumerate(contexts)])

    prompt = f"""
You are a helpful assistant.
Answer the question using only the provided context.
If the answer is not clearly supported by the context, say you do not know.

Question:
{query}

Context:
{context_text}
"""

    response = openai_client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": "You answer based only on retrieved context."},
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    return response.choices[0].message.content, result

In [26]:
# RAG 실행
answer, result = generate_rag_answer(
    query="Why is sparse-view CT difficult to reconstruct?",
    namespace="research",
    top_k=3
)

print("Answer:\n")
print(answer)

print("\nRetrieved contexts:\n")
for i, match in enumerate(result["matches"], 1):
    print(f"[{i}] score={match['score']:.4f}")
    print(match["metadata"]["text"])
    print("-" * 80)

Answer:

Sparse-view CT is difficult to reconstruct because only a limited number of projection angles are available, which makes the reconstruction problem more ill-posed.

Retrieved contexts:

[1] score=0.7427
In sparse-view CT, only a limited number of projection angles are available. This reduces radiation dose, but makes reconstruction more ill-posed. Regularization and learned priors
--------------------------------------------------------------------------------
[2] score=0.2847
Pinecone supports dense and sparse indexes. Dense indexes are used for semantic search, while sparse indexes are useful for lexical search. Namespaces can isolate data for multiten
--------------------------------------------------------------------------------
[3] score=0.2622
ularization and learned priors are commonly used to suppress artifacts.
--------------------------------------------------------------------------------


In [27]:
# 실험 과제
experiments = [
    {
        "query": "What is a vector database?",
        "namespace": "research",
        "top_k": 2,
        "filter": None
    },
    {
        "query": "What is a vector database?",
        "namespace": "teaching",
        "top_k": 2,
        "filter": None
    },
    {
        "query": "How does metadata filtering help in RAG?",
        "namespace": "research",
        "top_k": 3,
        "filter": {"topic": {"$eq": "rag"}}
    }
]

for exp in experiments:
    print("=" * 100)
    print(exp)
    ans, res = generate_rag_answer(
        query=exp["query"],
        namespace=exp["namespace"],
        top_k=exp["top_k"],
        metadata_filter=exp["filter"]
    )
    print("ANSWER:", ans)
    print()

{'query': 'What is a vector database?', 'namespace': 'research', 'top_k': 2, 'filter': None}
ANSWER: The provided context does not clearly define what a vector database is. Therefore, I do not know based on the given information.

{'query': 'What is a vector database?', 'namespace': 'teaching', 'top_k': 2, 'filter': None}
ANSWER: A vector database stores embeddings and retrieves semantically similar items.

{'query': 'How does metadata filtering help in RAG?', 'namespace': 'research', 'top_k': 3, 'filter': {'topic': {'$eq': 'rag'}}}
ANSWER: Metadata filtering helps in RAG by restricting retrieval to a subset of documents, such as papers only, lecture notes only, or a specific domain. This often improves precision in practical RAG systems.



# LangChain + Pinecone

LangChain과 Pinecone을 연결하여
문서를 임베딩하고, 벡터 데이터베이스에 저장한 뒤,
질문과 관련된 문서를 검색하는 기본 RAG 파이프라인을 구현.

## 왜 Pinecone을 쓰는가?

Pinecone은 클라우드 기반의 고성능 벡터 데이터베이스이다.
대규모 임베딩 데이터를 빠르게 저장하고 검색할 수 있어
실제 서비스형 RAG 시스템에 적합하다.

반면, FAISS나 Chroma는 무료이자 오픈소스라서 실습과 연구에는 매우 좋지만,
데이터 규모가 커질수록 관리, 확장성, 운영 측면에서 한계가 있을 수 있다.

## 왜 LangChain과 같이 쓰는가?

LangChain은 문서 로딩, 텍스트 분할, 임베딩, 검색기(retriever),
질의응답 체인 구성을 모듈화해서 쉽게 연결할 수 있게 해준다.

즉,
- Pinecone은 벡터 저장 및 검색을 담당하고
- LangChain은 전체 RAG 흐름을 연결하는 역할을 한다.

In [1]:
# 패키지 설치
!pip install -U "numpy<2" \
    langchain \
    langchain-core \
    langchain-openai \
    langchain-pinecone \
    langchain-text-splitters \
    pinecone

  Using cached pinecone-8.1.2-py3-none-any.whl.metadata (14 kB)


In [2]:
# API key 입력
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")
os.environ["PINECONE_API_KEY"] = getpass("Pinecone API Key: ")

OpenAI API Key: ··········
Pinecone API Key: ··········


In [3]:
# 기본 import
import time
from pinecone import Pinecone, ServerlessSpec

from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_pinecone import PineconeVectorStore

In [4]:
# 실습용 문서 준비
docs = [
    Document(
        page_content="""
        Computed tomography reconstructs cross-sectional images from projection data.
        Filtered back projection is fast, but noisy or sparse-view data can create artifacts.
        Iterative reconstruction can improve image quality, although it is more computationally expensive.
        """,
        metadata={"topic": "ct", "source": "lecture", "level": "basic"}
    ),
    Document(
        page_content="""
        Sparse-view CT uses fewer projection angles to reduce radiation dose.
        However, this makes the inverse problem more ill-posed and increases streak artifacts.
        Regularization and learned priors are commonly used to improve reconstruction quality.
        """,
        metadata={"topic": "ct", "source": "paper", "level": "intermediate"}
    ),
    Document(
        page_content="""
        Retrieval-augmented generation retrieves relevant document chunks first
        and then provides them as context to a language model.
        The quality of chunking and retrieval strongly affects final answer quality.
        """,
        metadata={"topic": "rag", "source": "lecture", "level": "basic"}
    ),
    Document(
        page_content="""
        Pinecone is a cloud-based high-performance vector database.
        It is designed for scalable semantic search and retrieval workloads.
        Namespaces and metadata filters are useful for practical production systems.
        """,
        metadata={"topic": "pinecone", "source": "docs", "level": "intermediate"}
    ),
]

In [5]:
# 문서 청킹
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

split_docs = text_splitter.split_documents(docs)

print("원본 문서 수:", len(docs))
print("청킹 후 문서 수:", len(split_docs))
print(split_docs[0])

원본 문서 수: 4
청킹 후 문서 수: 4
page_content='Computed tomography reconstructs cross-sectional images from projection data.
        Filtered back projection is fast, but noisy or sparse-view data can create artifacts.
        Iterative reconstruction can improve image quality, although it is more computationally expensive.' metadata={'topic': 'ct', 'source': 'lecture', 'level': 'basic'}


In [6]:
# 임베딩 모델 준비
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [7]:
# 임베딩 차원 확인
sample_vec = embeddings.embed_query("hello pinecone")
embedding_dim = len(sample_vec)
print("embedding_dim:", embedding_dim)

embedding_dim: 1536


In [8]:
# Pinecone 클라이언트 및 인덱스 생성
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

index_name = "langchain-pinecone-colab-demo"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        vector_type="dense",
        dimension=embedding_dim,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        ),
        deletion_protection="disabled"
    )
    print("Index created.")
else:
    print("Index already exists.")

Index created.


In [9]:
# 인덱스 준비 대기
while True:
    info = pc.describe_index(index_name)
    is_ready = info.status["ready"]
    print("ready:", is_ready)
    if is_ready:
        break
    time.sleep(5)

ready: True


In [10]:
# LangChain의 PineconeVectorStore 생성
##LangChain 문서에 따르면 PineconeVectorStore는 기존 인덱스 이름과 Embeddings 객체를 받아 초기화할 수 있음

vectorstore = PineconeVectorStore.from_documents(
    documents=split_docs,
    index_name=index_name,
    embedding=embeddings,
    namespace="demo-namespace"
)

print("업로드 완료")

업로드 완료


In [11]:
#similarity search 테스트
results = vectorstore.similarity_search(
    query="Why is sparse-view CT difficult?",
    k=3
)

for i, doc in enumerate(results, 1):
    print(f"[{i}]")
    print("metadata:", doc.metadata)
    print("content:", doc.page_content)
    print("-" * 80)

[1]
metadata: {'level': 'intermediate', 'source': 'docs', 'topic': 'pinecone'}
content: Pinecone is a cloud-based high-performance vector database.
        It is designed for scalable semantic search and retrieval workloads.
        Namespaces and metadata filters are useful for practical production systems.
--------------------------------------------------------------------------------
[2]
metadata: {'level': 'basic', 'source': 'lecture', 'topic': 'ct'}
content: Computed tomography reconstructs cross-sectional images from projection data.
        Filtered back projection is fast, but noisy or sparse-view data can create artifacts.
        Iterative reconstruction can improve image quality, although it is more computationally expensive.
--------------------------------------------------------------------------------
[3]
metadata: {'level': 'intermediate', 'source': 'paper', 'topic': 'ct'}
content: Sparse-view CT uses fewer projection angles to reduce radiation dose.
        However, t

In [12]:
# similarity search with score
results_with_score = vectorstore.similarity_search_with_score(
    query="What does Pinecone do in RAG?",
    k=3
)

for i, (doc, score) in enumerate(results_with_score, 1):
    print(f"[{i}] score={score}")
    print("metadata:", doc.metadata)
    print("content:", doc.page_content)
    print("-" * 80)

[1] score=0.187125161
metadata: {'level': 'basic', 'source': 'lecture', 'topic': 'rag'}
content: Retrieval-augmented generation retrieves relevant document chunks first
        and then provides them as context to a language model.
        The quality of chunking and retrieval strongly affects final answer quality.
--------------------------------------------------------------------------------
[2] score=0.513284385
metadata: {'level': 'intermediate', 'source': 'docs', 'topic': 'pinecone'}
content: Pinecone is a cloud-based high-performance vector database.
        It is designed for scalable semantic search and retrieval workloads.
        Namespaces and metadata filters are useful for practical production systems.
--------------------------------------------------------------------------------
[3] score=0.137717515
metadata: {'level': 'intermediate', 'source': 'paper', 'topic': 'ct'}
content: Sparse-view CT uses fewer projection angles to reduce radiation dose.
        However, this 

In [13]:
# Retriever 만들기
##LangChain에서 VectorStore는 retriever로 바꿔서 RAG 체인에 붙일 수 있음

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

retrieved_docs = retriever.invoke("Explain retrieval-augmented generation.")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"[{i}]")
    print(doc.metadata)
    print(doc.page_content)
    print("-" * 80)

[1]
{'level': 'basic', 'source': 'lecture', 'topic': 'rag'}
Retrieval-augmented generation retrieves relevant document chunks first
        and then provides them as context to a language model.
        The quality of chunking and retrieval strongly affects final answer quality.
--------------------------------------------------------------------------------
[2]
{'level': 'basic', 'source': 'lecture', 'topic': 'ct'}
Computed tomography reconstructs cross-sectional images from projection data.
        Filtered back projection is fast, but noisy or sparse-view data can create artifacts.
        Iterative reconstruction can improve image quality, although it is more computationally expensive.
--------------------------------------------------------------------------------
[3]
{'level': 'intermediate', 'source': 'paper', 'topic': 'ct'}
Sparse-view CT uses fewer projection angles to reduce radiation dose.
        However, this makes the inverse problem more ill-posed and increases streak ar

In [14]:
# metadata filter 적용 검색

filtered_retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 3,
        "filter": {"topic": "ct"}
    }
)

filtered_docs = filtered_retriever.invoke("How can artifacts appear in CT reconstruction?")

for i, doc in enumerate(filtered_docs, 1):
    print(f"[{i}]")
    print(doc.metadata)
    print(doc.page_content)
    print("-" * 80)

[1]
{'level': 'basic', 'source': 'lecture', 'topic': 'ct'}
Computed tomography reconstructs cross-sectional images from projection data.
        Filtered back projection is fast, but noisy or sparse-view data can create artifacts.
        Iterative reconstruction can improve image quality, although it is more computationally expensive.
--------------------------------------------------------------------------------
[2]
{'level': 'intermediate', 'source': 'paper', 'topic': 'ct'}
Sparse-view CT uses fewer projection angles to reduce radiation dose.
        However, this makes the inverse problem more ill-posed and increases streak artifacts.
        Regularization and learned priors are commonly used to improve reconstruction quality.
--------------------------------------------------------------------------------


In [15]:
# 간단한 RAG 응답 생성

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

query = "Why is sparse-view CT hard to reconstruct?"

retrieved_docs = retriever.invoke(query)
context = "\n\n".join([doc.page_content for doc in retrieved_docs])

prompt = f"""
You are a helpful assistant.
Answer the question using only the context below.
If the context is insufficient, say you do not know.

Question:
{query}

Context:
{context}
"""

response = llm.invoke(prompt)
print(response.content)

Sparse-view CT is hard to reconstruct because using fewer projection angles makes the inverse problem more ill-posed and increases streak artifacts. This reduced data leads to challenges in accurately reconstructing the image, requiring regularization and learned priors to improve reconstruction quality.


# LlamaIndex + Pinecone

LlamaIndex와 Pinecone을 이용하여 문서를 읽고, 노드(node) 단위로 분할하고, 임베딩한 뒤 Pinecone 벡터 데이터베이스에 저장하여 검색 기반 질의응답(RAG)을 구현한다.

## 왜 LlamaIndex를 쓰는가?

LlamaIndex는 문서 중심 RAG 파이프라인을 쉽게 구성할 수 있게 해주는 프레임워크이다.
특히 문서 로딩, 노드 분할, 인덱스 생성, retriever/query engine 연결이 자연스럽다.

## 왜 Pinecone을 같이 쓰는가?

Pinecone은 클라우드 기반 벡터 데이터베이스로,
대규모 임베딩 데이터를 빠르게 저장하고 검색하는 데 적합하다.
특히 serverless index, namespace, metadata 관리 기능을 제공하여
실전형 RAG 시스템으로 확장하기 좋다.

In [16]:
!pip -q uninstall -y numpy llama-index llama-index-core llama-index-vector-stores-pinecone llama-index-readers-file pinecone

!pip -q install -U "numpy<2" \
    "pinecone[grpc]>=5.1.0,<6" \
    "llama-index>=0.11,<0.12" \
    "llama-index-vector-stores-pinecone>=0.2,<0.3" \
    "llama-index-readers-file>=0.2,<0.3" \
    pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 427.3/427.3 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.8/295.8 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.4/216.4 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.6/263.6 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is th

In [17]:
# API key 입력
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")
os.environ["PINECONE_API_KEY"] = getpass("Pinecone API Key: ")

OpenAI API Key: ··········
Pinecone API Key: ··········


In [18]:
import os
import time
from pathlib import Path

from pinecone.grpc import PineconeGRPC
from pinecone import ServerlessSpec

from llama_index.core import VectorStoreIndex, Document
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.node_parser import SemanticSplitterNodeParser
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.vector_stores.pinecone import PineconeVectorStore

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'validate_default' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'validate_default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(


In [19]:
# 실습용 문서
documents = [
    Document(
        text="""
        Computed tomography reconstructs cross-sectional images from projection data.
        Filtered back projection is fast and widely used, but sparse-view or noisy measurements can cause streak artifacts.
        Iterative reconstruction methods can improve image quality, although they are computationally more expensive.
        """,
        metadata={"topic": "ct", "source": "lecture", "level": "basic"}
    ),
    Document(
        text="""
        Sparse-view CT uses fewer projection angles in order to reduce radiation dose.
        However, the inverse problem becomes more ill-posed, and prior information or regularization is often required.
        Learned priors and optimization-based methods are widely studied in this area.
        """,
        metadata={"topic": "ct", "source": "paper", "level": "intermediate"}
    ),
    Document(
        text="""
        Retrieval-augmented generation first retrieves relevant chunks from an external knowledge source,
        then passes those chunks to a language model as context.
        The quality of chunking, embedding, and retrieval strongly affects answer quality.
        """,
        metadata={"topic": "rag", "source": "lecture", "level": "basic"}
    ),
    Document(
        text="""
        Pinecone is a cloud-based vector database designed for semantic search and retrieval workloads.
        Namespaces help isolate data, and metadata filtering helps restrict search to relevant subsets of documents.
        """,
        metadata={"topic": "pinecone", "source": "docs", "level": "intermediate"}
    ),
]

In [20]:
# 임베딩 모델 준비
## Pinecone의 공식 LlamaIndex 예시는 OpenAI 임베딩을 사용하며, 같은 임베딩 모델을 문서 분할용 SemanticSplitter와 벡터화에 함께 사용

embed_model = OpenAIEmbedding(
    model="text-embedding-3-small",
    api_key=os.environ["OPENAI_API_KEY"]
)

In [21]:
# 차원 확인
sample_vec = embed_model.get_text_embedding("hello pinecone")
embedding_dim = len(sample_vec)
print("embedding_dim:", embedding_dim)

embedding_dim: 1536


In [25]:
# Pinecone 인덱스 생성
from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

index_name = "llamaindex-pinecone-colab-demo"

existing = [idx["name"] for idx in pc.list_indexes()]

if index_name not in existing:
    pc.create_index(
        name=index_name,
        dimension=embedding_dim,
        metric="cosine",
        spec={
            "serverless": {
                "cloud": "aws",
                "region": "us-east-1"
            }
        },
        deletion_protection="disabled"
    )
    print("Index created.")
else:
    print("Index already exists.")

Index created.


In [26]:
while True:
    info = pc.describe_index(index_name)
    ready = info.status["ready"]
    print("ready:", ready)
    if ready:
        break
    time.sleep(5)

ready: True


In [27]:
# PineconeVectorStore 연결
pinecone_index = pc.Index(index_name)
vector_store = PineconeVectorStore(pinecone_index=pinecone_index)

In [28]:
# IngestionPipeline으로 문서 분할 + 임베딩 + 업서트

pipeline = IngestionPipeline(
    transformations=[
        SemanticSplitterNodeParser(
            buffer_size=1,
            breakpoint_percentile_threshold=95,
            embed_model=embed_model,
        ),
        embed_model,
    ],
    vector_store=vector_store,
)

nodes = pipeline.run(documents=documents)
print("생성된 node 수:", len(nodes))

Upserted vectors:   0%|          | 0/6 [00:00<?, ?it/s]

생성된 node 수: 6


In [29]:
stats = pinecone_index.describe_index_stats()
print(stats)

{'dimension': 1536,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 6}},
 'total_vector_count': 6}


In [30]:
# Pinecone에 저장된 데이터를 LlamaIndex로 다시 연결

vector_index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store,
    embed_model=embed_model
)

In [31]:
# Retriever로 검색

retriever = VectorIndexRetriever(
    index=vector_index,
    similarity_top_k=3
)

results = retriever.retrieve("Why is sparse-view CT difficult to reconstruct?")

for i, node in enumerate(results, 1):
    print(f"[{i}]")
    print(node.get_content())
    print("-" * 80)

[1]

        Sparse-view CT uses fewer projection angles in order to reduce radiation dose.
        However, the inverse problem becomes more ill-posed, and prior information or regularization is often required.
        
--------------------------------------------------------------------------------
[2]

        Computed tomography reconstructs cross-sectional images from projection data.
        Filtered back projection is fast and widely used, but sparse-view or noisy measurements can cause streak artifacts.
        
--------------------------------------------------------------------------------
[3]
Iterative reconstruction methods can improve image quality, although they are computationally more expensive.
        
--------------------------------------------------------------------------------


In [33]:
# Query Engine으로 간단한 RAG 답변 생성
llm = OpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=os.environ["OPENAI_API_KEY"]
)

query_engine = vector_index.as_query_engine(
    llm=llm,
    similarity_top_k=3
)

response = query_engine.query("Explain why sparse-view CT is challenging.")
print(response)

Sparse-view CT is challenging because it utilizes fewer projection angles, which leads to an ill-posed inverse problem. This situation often necessitates the use of prior information or regularization techniques to achieve accurate image reconstruction. Additionally, the limited data can result in artifacts, particularly when combined with noise, making it difficult to obtain high-quality images.


In [34]:
# 메타데이터를 확인하면서 검색하기

response = query_engine.query("What is Pinecone used for in RAG?")

print("답변:\n")
print(response)

print("\n참고 노드:\n")
for i, n in enumerate(response.source_nodes, 1):
    print(f"[{i}] score={n.score}")
    print("metadata:", n.node.metadata)
    print("text:", n.node.get_content())
    print("-" * 80)

답변:

Pinecone is utilized in retrieval-augmented generation (RAG) to manage and retrieve relevant chunks of data from a vector database, enhancing the semantic search and retrieval process. This allows for effective isolation of data and filtering based on metadata, which contributes to the quality of the information passed to the language model as context.

참고 노드:

[1] score=0.541701
metadata: {'topic': 'pinecone', 'source': 'docs', 'level': 'intermediate'}
text: 
        Pinecone is a cloud-based vector database designed for semantic search and retrieval workloads.
        Namespaces help isolate data, and metadata filtering helps restrict search to relevant subsets of documents.
        
--------------------------------------------------------------------------------
[2] score=0.3235486
metadata: {'topic': 'rag', 'source': 'lecture', 'level': 'basic'}
text: 
        Retrieval-augmented generation first retrieves relevant chunks from an external knowledge source,
        then passes 